In [2]:
import pandas as pd
import numpy as np

In [3]:
file_path = "/kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path)

# **PHASE 1 - Data Audit**

In [4]:
df.shape

(6362620, 11)

In [5]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [6]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


### Check null values

In [8]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

### Check Duplicates

In [9]:
df.duplicated().sum()

np.int64(0)

### Transaction types

In [11]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


### Fraud distribution

In [10]:
fraud_counts = df["isFraud"].value_counts()
fraud_percentages = df["isFraud"].value_counts(normalize=True) * 100

print("Counts:")
print(fraud_counts)

print("\nPercentages:")
print(fraud_percentages)

Counts:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Percentages:
isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64


In [12]:
pd.crosstab(df["type"], df["isFraud"])

isFraud,0,1
type,,
CASH_IN,1399284,0
CASH_OUT,2233384,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,528812,4097


In [13]:
fraud_by_type = (
    df.groupby("type")["isFraud"]
    .agg(["count", "sum", "mean"])
    .sort_values("mean", ascending=False)
)

fraud_by_type

,count,sum,mean
type,,,
TRANSFER,532909,4097,0.007688
CASH_OUT,2237500,4116,0.001840
CASH_IN,1399284,0,0.000000
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000


In [14]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
step,6362620.0,2.433972e+02,1.423320e+02,1.0,156.00,239.000,3.350000e+02,7.430000e+02
amount,6362620.0,1.798619e+05,6.038582e+05,0.0,13389.57,74871.940,2.087215e+05,9.244552e+07
oldbalanceOrg,6362620.0,8.338831e+05,2.888243e+06,0.0,0.00,14208.000,1.073152e+05,5.958504e+07
newbalanceOrig,6362620.0,8.551137e+05,2.924049e+06,0.0,0.00,0.000,1.442584e+05,4.958504e+07
oldbalanceDest,6362620.0,1.100702e+06,3.399180e+06,0.0,0.00,132705.665,9.430367e+05,3.560159e+08
newbalanceDest,6362620.0,1.224996e+06,3.674129e+06,0.0,0.00,214661.440,1.111909e+06,3.561793e+08
isFraud,6362620.0,1.290820e-03,3.590480e-02,0.0,0.00,0.000,0.000000e+00,1.000000e+00
isFlaggedFraud,6362620.0,2.514687e-06,1.585775e-03,0.0,0.00,0.000,0.000000e+00,1.000000e+00


# **PHASE 2 - Leakage decisions**
## Feature Selection and Leakage Considerations

The target variable is `isFraud`.

The following fields are excluded from the initial model:

- `nameOrig` and `nameDest`: account identifiers rather than meaningful generalizable behavioral features.
- `isFlaggedFraud`: an existing fraud-rule output that may leak information from another detection mechanism.
- `newbalanceOrig` and `newbalanceDest`: post-transaction balances that may introduce information unavailable or inappropriate at prediction time.

The initial model therefore uses transaction information that can reasonably represent the transaction at scoring time.

# **PHASE 3 - Create modeling dataset**

In [16]:
features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest"
]

X = df[features]
y = df["isFraud"]

print(X.shape)
print(y.shape)

(6362620, 5)
(6362620,)


In [17]:
from sklearn.model_selection import train_test_split

X_work, _, y_work, _ = train_test_split(
    X,
    y,
    train_size=800_000,
    stratify=y,
    random_state=42
)

print("Working sample shape:", X_work.shape)
print("\nFraud distribution:")
print(y_work.value_counts())
print("\nFraud percentage:")
print(y_work.value_counts(normalize=True) * 100)

Working sample shape: (800000, 5)

Fraud distribution:
isFraud
0    798967
1      1033
Name: count, dtype: int64

Fraud percentage:
isFraud
0    99.870875
1     0.129125
Name: proportion, dtype: float64


# **PHASE 4 - Train / validation / test**

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_work,
    y_work,
    test_size=0.30,
    stratify=y_work,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nFraud rates:")
print("Train:", y_train.mean())
print("Validation:", y_val.mean())
print("Test:", y_test.mean())

Train: (560000, 5)
Validation: (120000, 5)
Test: (120000, 5)

Fraud rates:
Train: 0.0012910714285714285
Validation: 0.0012916666666666667
Test: 0.0012916666666666667
